# CAREPILOT — Transfer Prediction Benchmark

*Chapter 2 — État de l'art et benchmarking IA · CAREPILOT operational AI*

## 1. Objective

Predict whether a ticket requires **cross-perimeter transfer** (`transfer_required`) to another operational team.

## 2. Problem Type

- **Binary classification**
- Target: `transfer_required`
- Primary metrics: F1-score, Recall, ROC-AUC

## 3. Dataset Description


- **Source:** `ml_training.historical_tickets`
- **Records:** 14,000 tickets
- **Target:** `transfer_required` (~35% positive)
- **Features:** title, description, perimeter, priority, SLA breach, resolution time, incident category

## 4. Preprocessing

Same CAREPILOT text + metadata pipeline as Task 1 (TF-IDF, one-hot, scaling, stratified 70/15/15 split).

## 5. Models / Methods Compared

Logistic Regression · Random Forest · XGBoost

This benchmark compares multiple approaches in order to select the most suitable method for integration into CAREPILOT.

## 6. Evaluation Metrics

Accuracy · Precision · Recall · F1 · ROC-AUC · CV accuracy · Training/inference time

## 7. Training or Retrieval Pipeline

Identical sklearn pipeline structure to escalation prediction with `transfer_required` as target.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image, Markdown, display

NOTEBOOK_DIR = Path.cwd()
ROOT = NOTEBOOK_DIR
for parent in [NOTEBOOK_DIR, *NOTEBOOK_DIR.parents]:
    if (parent / "ml" / "artifacts").exists():
        ROOT = parent
        break

SHARED = ROOT / "ml" / "notebooks" / "_shared"
if str(SHARED) not in sys.path:
    sys.path.insert(0, str(SHARED))

from benchmark_notebook_helpers import display_plot, find_project_root, plot_results_table

ROOT = find_project_root()
ARTIFACTS = ROOT / "ml" / "artifacts"
PLOTS = ROOT / "ml" / "plots"
REPORTS = ROOT / "ml" / "reports"
MODELS = ARTIFACTS / "models"

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 20)
pd.set_option("display.float_format", lambda x: f"{x:.4f}")

RUN_BENCHMARK = False  # Set True to re-train / re-run the full benchmark pipeline


In [ ]:
BENCHMARK_SCRIPT = ROOT / "ml/benchmarks/transfer/benchmark.py"
if RUN_BENCHMARK:
    cmd = [sys.executable, str(BENCHMARK_SCRIPT), "--project-root", str(ROOT), *[]]
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, check=True)
else:
    print("Skipping benchmark run (RUN_BENCHMARK=False). Loading saved artifacts.")
    print("Re-run with RUN_BENCHMARK=True or execute:")
    print(f"  python {BENCHMARK_SCRIPT} --project-root {ROOT}")


## 8. Benchmark Execution — Load Results

In [ ]:
metrics_path = ARTIFACTS / "03_transfer_benchmark_metrics.json"
if not metrics_path.exists():
    raise FileNotFoundError(f"Missing metrics artifact: {metrics_path}")
payload = json.loads(metrics_path.read_text(encoding="utf-8"))
rows = payload if isinstance(payload, list) else payload.get("results", payload.get("models", []))
results_df = pd.DataFrame(rows)
cols = [c for c in ["model", "Model", "accuracy", "Accuracy", "precision", "recall", "f1", "roc_auc", "ROC-AUC"] if c in results_df.columns]
if cols:
    results_df = results_df[cols]
display(results_df)


## 9. Benchmark Results

Final comparison table (report-ready PNG + DataFrame):

In [ ]:
table_png = PLOTS / "03_transfer_benchmark_results_table.png"
display_plot(table_png)
display_plot(PLOTS / "03_transfer_model_comparison.png")
display_plot(PLOTS / "03_transfer_roc_curves.png")
display_plot(PLOTS / "03_transfer_distribution.png")


## 10. Visualizations

Key plots are loaded from `ml/plots/` (generated by the benchmark pipeline). Re-run with `RUN_BENCHMARK=True` to regenerate.

## 11. Interpretation


- **Logistic Regression:** moderate recall (~49%) — explainable but not sufficient for routing.
- **Random Forest:** balanced precision/recall (~71% F1).
- **XGBoost:** best F1 (~0.78) and recall (~81%) — minimizes missed transfer candidates.

Cross-perimeter symptom overlap and inconsistent operator notes create realistic ambiguity in the dataset.

## 12. Selected Model / Method

**Selected for CAREPILOT:**

**XGBoost** — highest F1 and recall. Production: `xgboost_transfer.joblib`.

## 13. Limitations

- Misleading transfer wording on locally resolved tickets
- Perimeter overlap increases false positives
- Label noise from historical misroutes

## 14. Generated Artifacts

- `ml/artifacts/03_transfer_benchmark_metrics.json`
- `ml/plots/03_transfer_benchmark_results_table.png`
- `ml/reports/03_transfer_benchmark_summary.md`